In [ ]:
import pandas as pd
import numpy as np

product_url = "https://drive.google.com/uc?id=11jvH4R2ntNmWWofuD-qebHmyWOHtG0nu"
payment_report_url = "https://drive.google.com/uc?id=1hqx4-3cFerSElpij1YWgGusWVtldvrBr"
transactions_url = "https://drive.google.com/uc?id=1c1nCpuACsJBklJ9YpoiihATj9JkiLJFM"

product = pd.read_csv(product_url)
payment_report = pd.read_csv(payment_report_url)
transactions = pd.read_csv(transactions_url)

Given dataset
Suppose you are a DA in an e-wallet company, and you need to analyze the following datasets:
payment_report.csv (monthly payment volume of products)
product.csv (product information)
transactions.csv (transactions information)

**Statement: Muốn hiểu về tình trạng payment hoặc transaction trong context của một ewallet.**

**Part I: EDA - Explore Data Analysis**
Do EDA task:
- Df payment_enriched (Merge payment_report.csv with product.csv)
- Df transactions
Suggestions:
1. Check each column: missing data? duplicates? incorrect data types?
2. Summarize numerical data: any incorrect values?

Sample Answers:
- Incorrect data types: column A, column B -> Next step: No action/ Delete rows/…
- Incorrect values: column A, column B -> Next step: No action/ Delete rows/…
- Missing data: x rows in column A, y rows in column B -> Next step: No action/ Delete rows/…
- Duplicates: PK? x rows? -> Next step: No action/ Delete rows/…

**Part II: Data Wrangling**

Using payment_report.csv & product.csv
1. Top 3 product_ids with the highest volume.
2. Given that 1 product_id is only owed by 1 team, are there any abnormal products against this rule?
3. Find the team has had the lowest performance (lowest volume) since Q2.2023 Find the category that contributes the least to that team.
4. Find the contribution of source_ids of refund transactions (payment_group = ‘refund’), what is the source_id with the highest contribution?

Using transactions.csv

5. Define type of transactions (‘transaction_type’) for each row, given:
- transType = 2 & merchant_id = 1205: Bank Transfer Transaction
- transType = 2 & merchant_id = 2260: Withdraw Money Transaction
- transType = 2 & merchant_id = 2270: Top Up Money Transaction
- transType = 2 & others merchant_id: Payment Transaction
- transType = 8, merchant_id = 2250: Transfer Money Transaction
- transType = 8 & others merchant_id: Split Bill Transaction
- Remained cases are invalid transactions
6. Of each transaction type (excluding invalid transactions): find the number of transactions, volume, senders and receivers.


---

**Part I: EDA - Explore Data Analysis**
Do EDA task:
- Df payment_enriched (Merge payment_report.csv with product.csv)
- Df transactions
Suggestions:
1. Check each column: missing data? duplicates? incorrect data types?
2. Summarize numerical data: any incorrect values?

Sample Answers:
- Incorrect data types: column A, column B -> Next step: No action/ Delete rows/…
- Incorrect values: column A, column B -> Next step: No action/ Delete rows/…
- Missing data: x rows in column A, y rows in column B -> Next step: No action/ Delete rows/…
- Duplicates: PK? x rows? -> Next step: No action/ Delete rows/…

In [ ]:
# Df payment_enriched (Merge payment_report.csv with product.csv)

payment_enriched = payment_report.merge(product, on = 'product_id', validate= 'many_to_one', how = 'left')

In [ ]:
# Df transactions Suggestions:
# Check general info

print(transactions.info())
print(transactions.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1324002 entries, 0 to 1324001
Data columns (total 9 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   transaction_id  1324002 non-null  int64  
 1   merchant_id     1324002 non-null  int64  
 2   volume          1324002 non-null  int64  
 3   transType       1324002 non-null  int64  
 4   transStatus     1324002 non-null  int64  
 5   sender_id       1274943 non-null  float64
 6   receiver_id     1159207 non-null  float64
 7   extra_info      6095 non-null     object 
 8   timeStamp       1324002 non-null  int64  
dtypes: float64(2), int64(6), object(1)
memory usage: 90.9+ MB
None
   transaction_id  merchant_id   volume  transType  transStatus   sender_id  \
0      3002692434            5   100000         24            1  10199794.0   
1      3002692437          305    20000          2            1  14022211.0   
2      3001960110         7255    48605         22            1   

In [ ]:
transactions_remove_duplicate = transactions.drop_duplicates()  #remove duplicate rows
print(transactions_remove_duplicate.info())

<class 'pandas.core.frame.DataFrame'>
Index: 1323974 entries, 0 to 1324001
Data columns (total 9 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   transaction_id  1323974 non-null  int64  
 1   merchant_id     1323974 non-null  int64  
 2   volume          1323974 non-null  int64  
 3   transType       1323974 non-null  int64  
 4   transStatus     1323974 non-null  int64  
 5   sender_id       1274916 non-null  float64
 6   receiver_id     1159182 non-null  float64
 7   extra_info      6095 non-null     object 
 8   timeStamp       1323974 non-null  int64  
dtypes: float64(2), int64(6), object(1)
memory usage: 101.0+ MB
None


In [ ]:
print(transactions_remove_duplicate.isnull().sum())   #Check null values

transaction_id          0
merchant_id             0
volume                  0
transType               0
transStatus             0
sender_id           49058
receiver_id        164792
extra_info        1317879
timeStamp               0
dtype: int64


In [ ]:
transactions_new = transactions_remove_duplicate.drop(columns = 'extra_info')  #Remove 'extra_info' column
print(transactions_new.info())

<class 'pandas.core.frame.DataFrame'>
Index: 1323974 entries, 0 to 1324001
Data columns (total 8 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   transaction_id  1323974 non-null  int64  
 1   merchant_id     1323974 non-null  int64  
 2   volume          1323974 non-null  int64  
 3   transType       1323974 non-null  int64  
 4   transStatus     1323974 non-null  int64  
 5   sender_id       1274916 non-null  float64
 6   receiver_id     1159182 non-null  float64
 7   timeStamp       1323974 non-null  int64  
dtypes: float64(2), int64(6)
memory usage: 90.9 MB
None


In [ ]:
transactions_new[['sender_id','receiver_id']] = transactions_new[['sender_id','receiver_id']].fillna(0).astype('int') # Fillna =0 in sender_id & receive_id and change type to integer
print(transactions_new.info())

<class 'pandas.core.frame.DataFrame'>
Index: 1323974 entries, 0 to 1324001
Data columns (total 8 columns):
 #   Column          Non-Null Count    Dtype
---  ------          --------------    -----
 0   transaction_id  1323974 non-null  int64
 1   merchant_id     1323974 non-null  int64
 2   volume          1323974 non-null  int64
 3   transType       1323974 non-null  int64
 4   transStatus     1323974 non-null  int64
 5   sender_id       1323974 non-null  int64
 6   receiver_id     1323974 non-null  int64
 7   timeStamp       1323974 non-null  int64
dtypes: int64(8)
memory usage: 90.9 MB
None


---

**Part II: Data Wrangling**

Using payment_report.csv & product.csv
1. Top 3 product_ids with the highest volume.
2. Given that 1 product_id is only owed by 1 team, are there any abnormal products against this rule?
3. Find the team has had the lowest performance (lowest volume) since Q2.2023 Find the category that contributes the least to that team.
4. Find the contribution of source_ids of refund transactions (payment_group = ‘refund’), what is the source_id with the highest contribution?

In [ ]:
print(payment_enriched.info())
print(payment_enriched.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 919 entries, 0 to 918
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   report_month   919 non-null    object
 1   payment_group  919 non-null    object
 2   product_id     919 non-null    int64 
 3   source_id      919 non-null    int64 
 4   volume         919 non-null    int64 
 5   category       897 non-null    object
 6   team_own       897 non-null    object
dtypes: int64(3), object(4)
memory usage: 50.4+ KB
None
  report_month payment_group  product_id  source_id     volume  category  \
0      2023-01       payment          12         45  624110375   PXXXXXT   
1      2023-01       payment          17         45  335715113   PXXXXXB   
2      2023-01       payment          18         45  737784466   PXXXXXB   
3      2023-01       payment          19         45  120963069  PXXXXXM2   
4      2023-01       payment          20         45  319653158   PXXXXXB 

In [ ]:
print(payment_enriched.groupby('product_id')['volume'].sum().sort_values(ascending = False).head(3))  # Top 3 product_ids with the highest volume.


product_id
1976    61797583647
429     14667676567
372     13713658515
Name: volume, dtype: int64


In [ ]:
# Given that 1 product_id is only owed by 1 team, are there any abnormal products against this rule?

groupteam = payment_enriched.groupby('product_id')['team_own'].nunique().sort_values(ascending = False) # Check if any produt owed by more than 1 team -> no product
print(groupteam[groupteam.index.isin([1976, 429, 372])])

#-> Product_id = 1976 has highest Volume but does not have any team_own

product_id
429     1
372     1
1976    0
Name: team_own, dtype: int64


In [ ]:
#Find the team has had the lowest performance (lowest volume) since Q2.2023 Find the category that contributes the least to that team
payment_enriched['report_date'] = pd.to_datetime(payment_enriched['report_month'])

print(payment_enriched[payment_enriched['report_date'] >= '2023-04-01'].groupby('team_own')['volume'].sum().sort_values(ascending = True).head(1))

# -> Team APS has the lowest volume since Q2.2023

print(payment_enriched[(payment_enriched['team_own'] == 'APS') & (payment_enriched['report_date'] >= '2023-04-01')].groupby('category')['volume'].sum().sort_values(ascending = True).head(1))

# -> category 'PXXXXXS' contributes the least to team APS

team_own
APS    51141753
Name: volume, dtype: int64
category
PXXXXXE    25232438
Name: volume, dtype: int64


In [ ]:
# Find the contribution of source_ids of refund transactions (payment_group = ‘refund’), what is the source_id with the highest contribution?

print(payment_enriched[payment_enriched['payment_group'] == 'refund'].groupby('source_id')['volume'].sum().sort_values(ascending = False).head(1))

# -> source_id 38 has the highest contribution within payment_group 'refund'



source_id
38    36527454759
Name: volume, dtype: int64


Using transactions.csv

5. Define type of transactions (‘transaction_type’) for each row, given:
- transType = 2 & merchant_id = 1205: Bank Transfer Transaction
- transType = 2 & merchant_id = 2260: Withdraw Money Transaction
- transType = 2 & merchant_id = 2270: Top Up Money Transaction
- transType = 2 & others merchant_id: Payment Transaction
- transType = 8, merchant_id = 2250: Transfer Money Transaction
- transType = 8 & others merchant_id: Split Bill Transaction
- Remained cases are invalid transactions
6. Of each transaction type (excluding invalid transactions): find the number of transactions, volume, senders and receivers.

In [ ]:
print(transactions_new.head())
print(transactions_new.isnull().sum())

   transaction_id  merchant_id   volume  transType  transStatus  sender_id  \
0      3002692434            5   100000         24            1   10199794   
1      3002692437          305    20000          2            1   14022211   
2      3001960110         7255    48605         22            1          0   
3      3002680710         2270  1500000          2            1   10059206   
4      3002680713         2275    90000          2            1   10004711   

   receiver_id      timeStamp  
0       199794  1682932054455  
1     14022211  1682932054912  
2     10530940  1682932055000  
3        59206  1682932055622  
4         4711  1682932056197  
transaction_id    0
merchant_id       0
volume            0
transType         0
transStatus       0
sender_id         0
receiver_id       0
timeStamp         0
dtype: int64


In [ ]:
condition = [((transactions_new['transType'] == 2) & (transactions_new['merchant_id'] == 1205)), \
             ((transactions_new['transType'] == 2) & (transactions_new['merchant_id'] == 2260)), \
             ((transactions_new['transType'] == 2) & (transactions_new['merchant_id'] == 2270)), \
             ((transactions_new['transType'] == 2) & (~transactions_new['merchant_id'].isin([1205, 2260, 2270]))), \
             ((transactions_new['transType'] == 8) & (transactions_new['merchant_id'] == 2250)), \
             ((transactions_new['transType'] == 8) & (transactions_new['merchant_id'] != 2250))]

result = ['Bank Transfer Transaction', 'Withdraw Money Transaction', 'Top Up Money Transaction', 'Payment Transaction', 'Transfer Money Transaction', 'Split Bill Transaction']

In [ ]:
# 5. Define type of transactions (‘transaction_type’) for each row

transactions_new['transaction_type'] = np.select(condition, result, default = 'Invalid Transaction')
print(transactions_new['transaction_type'].value_counts(ascending = False))

transaction_type
Payment Transaction           398665
Transfer Money Transaction    341173
Top Up Money Transaction      290498
Invalid Transaction           220658
Bank Transfer Transaction      37879
Withdraw Money Transaction     33725
Split Bill Transaction          1376
Name: count, dtype: int64


In [ ]:
# 6. Of each transaction type (excluding invalid transactions): find the number of transactions, volume, senders and receivers.
# -> count transaction_id, sum volume, count unique sender_id, count unique receiver_id

transactions_new[transactions_new['transaction_type'] != 'Invalid Transaction'].groupby('transaction_type').agg(total_transaction =('transaction_id','count'),
                                                                                                                total_volume = ('volume','sum'),
                                                                                                                total_unique_sender = ('sender_id','nunique'),
                                                                                                                total_unique_receiver = ('receiver_id','nunique'))

,total_transaction,total_volume,total_unique_sender,total_unique_receiver
transaction_type,,,,
Bank Transfer Transaction,37879,50605806190,23156,9272
Payment Transaction,398665,71850608441,139583,113298
Split Bill Transaction,1376,4901464,1323,572
Top Up Money Transaction,290498,108605618829,110409,110409
Transfer Money Transaction,341173,37032880492,39021,34585
Withdraw Money Transaction,33725,23418181420,24814,24814
